# 09.1 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/09-nlp-genai/01-text-to-numbers/solutions.ipynb)

In [1]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import os, logging, warnings
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, str(data_path("download.py").parent))

## Exercise 1

Vocabulary: *broken, lights, lovely, mug*.

| | broken | lights | lovely | mug |
|---|---|---|---|---|
| "lovely lights" | 0 | 1 | 1 | 0 |
| "lights broken" | 1 | 1 | 0 | 0 |
| "lovely lovely mug" | 0 | 0 | 2 | 1 |

$N = 3$. *lovely* is in 2 reviews: idf $= \ln(4/3) + 1 = 1.2877$. *mug* is in 1: idf $= \ln(4/2) + 1 = 1.6931$.

Third review raw: lovely $2 \times 1.2877 = 2.5754$, mug $1.6931$. Length $\sqrt{2.5754^2 + 1.6931^2} = 3.0821$. **lovely = 0.836**, mug = 0.549.

In [2]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

docs = ["lovely lights", "lights broken", "lovely lovely mug"]
cv = CountVectorizer()
print(pd.DataFrame(cv.fit_transform(docs).toarray(), columns=cv.get_feature_names_out()))
tv = TfidfVectorizer()
X = tv.fit_transform(docs)
pd.DataFrame(X.toarray().round(3), columns=tv.get_feature_names_out())

   broken  lights  lovely  mug
0       0       1       1    0
1       1       1       0    0
2       0       0       2    1


,broken,lights,lovely,mug
0,0.000,0.707,0.707,0.000
1,0.796,0.605,0.000,0.000
2,0.000,0.000,0.836,0.549


## Exercise 2

In [3]:
v = CountVectorizer(ngram_range=(2, 2)).fit(["the mug was not good at all"])
list(v.get_feature_names_out())

['at all', 'good at', 'mug was', 'not good', 'the mug', 'was not']

*not good* is the bigram that carries the sentiment. On its own, *good* points the wrong way.

## Exercise 3

In [4]:
list(CountVectorizer(stop_words="english").fit(["this is not good"]).get_feature_names_out())

['good']

Only *good* survives: *this*, *is* and *not* are all on the stop list. A sentiment model fed this would read a complaint as praise.

## Exercise 4

In [5]:
from download import load_reviews
from sklearn.model_selection import train_test_split

reviews = load_reviews()          # first run downloads one 117 MB file, then cached
train, test = train_test_split(reviews, test_size=0.2, random_state=42, stratify=reviews["label"])
print(f"{len(reviews):,} reviews ({reviews['label'].mean():.0%} positive): "
      f"train {len(train):,} / test {len(test):,}")
reviews[["text", "label"]].head()

10,000 reviews (50% positive): train 8,000 / test 2,000


,text,label
0,"Great Book, Must read! Ramon found the fabulou...",1
1,A leaker! This is the second generation of the...,0
2,Not For the Professional Cook. Henckels are we...,0
3,Arg! I'd like to see Robert Jordan put to the ...,0
4,"If I could give this minus 5 stars, I'd give i...",0


In [6]:
cv = CountVectorizer()
C = cv.fit_transform(reviews["text"])
neg = np.asarray(C[(reviews["label"] == 0).to_numpy()].sum(axis=0)).ravel()
total = np.asarray(C.sum(axis=0)).ravel()
words = pd.DataFrame({"uses": total, "negative share": neg / total},
                     index=cv.get_feature_names_out())
common = words[words["uses"] >= 50]
print(common.sort_values("negative share", ascending=False).head(10).round(3))
print(common.sort_values("negative share").head(10).round(3))

               uses  negative share
refund           80           0.988
awful           180           0.972
worthless        62           0.968
junk            146           0.959
waste           468           0.957
worst           280           0.957
warranty         93           0.957
disappointing   180           0.950
ridiculous       59           0.949
poorly          160           0.944
             uses  negative share
awesome       199           0.085
terrific       53           0.113
unique         70           0.114
wonderful     405           0.123
perfect       370           0.135
loves         181           0.138
excellent     547           0.150
brings         63           0.159
perspective    55           0.164
superb         67           0.164


The most negative words are complaints: *refund* (98.8% of its 80 uses are in negative reviews), *awful*, *worthless*, *junk*, *waste*. Notice *warranty*: people only mention it when something broke. The most positive are praise: *awesome* (only 8.5% negative), *terrific*, *unique*, *wonderful*, *perfect*, *excellent*. Counting alone already separates the moods: that is why 09.2's simple classifier works as well as it does.